# 현금 등 기타 지출 내역

In [1]:
from ledgerly.expenditure import (cash_file_config
                                  , preprocess_cash_data
                                  , map_cash_df_to_expenditure
                                  , map_category
                                  , insert_expenditure_data
                                  , fetch_expenditure_data)
from ledgerly.utils import find_project_root

In [2]:
import pandas as pd

## 1. 현금 파일로부터 데이터 불러오기

In [3]:
# 데이터 파일 경로 설정 (가변적)
target_yymm = "2604" # 예: "2603", 없을 경우 빈 문자열 ""
target_filename = "cash_form.csv"

if target_yymm:
    target_file = cash_file_config["data_dir"] / target_yymm / target_filename
else:
    target_file = cash_file_config["data_dir"] / target_filename

cash_df = pd.read_csv(target_file)


In [4]:
cash_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   이용일     20 non-null     object 
 1   사용처     20 non-null     object 
 2   금액      20 non-null     int64  
 3   결제수단    20 non-null     object 
 4   결제제공자   20 non-null     object 
 5   memo    0 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 1.1+ KB


In [5]:
cash_df.head()

,이용일,사용처,금액,결제수단,결제제공자,memo
0,2026-03-27,맑은하늘적금,200000,현금,KB,NaN
1,2026-03-23,DB손해보험,80330,현금,KB,NaN
2,2026-03-23,운전자보험,13130,현금,KB,NaN
3,2026-03-09,어린이보험,73000,현금,KB,NaN
4,2026-03-09,청약,100000,현금,KB,NaN


## 2. 현금 데이터 전처리

In [6]:
# 전처리 및 매핑 로직은 ledgerly.expenditure.cash 모듈로 이동되었습니다.

In [7]:
cash_df = preprocess_cash_data(cash_df)
cash_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   이용일     20 non-null     datetime64[ns]
 1   사용처     20 non-null     object        
 2   금액      20 non-null     int64         
 3   결제수단    20 non-null     object        
 4   결제제공자   20 non-null     object        
 5   memo    20 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 1.1+ KB


In [8]:
cash_df.head()

,이용일,사용처,금액,결제수단,결제제공자,memo
0,2026-03-27,맑은하늘적금,200000,현금,KB,nan
1,2026-03-23,DB손해보험,80330,현금,KB,nan
2,2026-03-23,운전자보험,13130,현금,KB,nan
3,2026-03-09,어린이보험,73000,현금,KB,nan
4,2026-03-09,청약,100000,현금,KB,nan


In [9]:
# 매핑 로직은 ledgerly.expenditure.cash 모듈로 이동되었습니다.

In [10]:
expenditure_df = map_cash_df_to_expenditure(cash_df)
expenditure_df.head()

,used_at,merchant_name,amount,payment_type,payment_provider,memo,category,installment_type,remaining_amount,source_uid
0,2026-03-27,맑은하늘적금,200000,현금,KB,nan,unknown,single,0,현금_KB_20260327_맑은하늘적금_nan
1,2026-03-23,DB손해보험,80330,현금,KB,nan,unknown,single,0,현금_KB_20260323_DB손해보험_nan
2,2026-03-23,운전자보험,13130,현금,KB,nan,unknown,single,0,현금_KB_20260323_운전자보험_nan
3,2026-03-09,어린이보험,73000,현금,KB,nan,unknown,single,0,현금_KB_20260309_어린이보험_nan
4,2026-03-09,청약,100000,현금,KB,nan,unknown,single,0,현금_KB_20260309_청약_nan


In [11]:
expenditure_df["category"] = expenditure_df["merchant_name"].apply(map_category)
expenditure_df.head()

,used_at,merchant_name,amount,payment_type,payment_provider,memo,category,installment_type,remaining_amount,source_uid
0,2026-03-27,맑은하늘적금,200000,현금,KB,nan,저축,single,0,현금_KB_20260327_맑은하늘적금_nan
1,2026-03-23,DB손해보험,80330,현금,KB,nan,보험,single,0,현금_KB_20260323_DB손해보험_nan
2,2026-03-23,운전자보험,13130,현금,KB,nan,보험,single,0,현금_KB_20260323_운전자보험_nan
3,2026-03-09,어린이보험,73000,현금,KB,nan,보험,single,0,현금_KB_20260309_어린이보험_nan
4,2026-03-09,청약,100000,현금,KB,nan,저축,single,0,현금_KB_20260309_청약_nan


In [12]:
insert_expenditure_data(expenditure_df)
fetch_expenditure_data()

,id,used_at,payment_type,payment_provider,merchant_name,installment_type,amount,remaining_amount,category,memo,source_uid,created_at
0,15,2026-01-31 00:00:00,현금,KB,경조사(용수),single,100000,0,경조사,nan,현금_KB_20260131_경조사(용수)_nan,2026-02-09 13:07:52
1,16,2026-01-27 00:00:00,현금,KB,맑은하늘적금,single,200000,0,저축,nan,현금_KB_20260127_맑은하늘적금_nan,2026-02-09 13:07:52
2,17,2026-01-21 00:00:00,현금,KB,DB손해보험,single,80330,0,보험,nan,현금_KB_20260121_DB손해보험_nan,2026-02-09 13:07:52
3,18,2026-01-21 00:00:00,현금,KB,운전자보험,single,13130,0,보험,nan,현금_KB_20260121_운전자보험_nan,2026-02-09 13:07:52
4,19,2026-01-09 00:00:00,현금,KB,어린이보험,single,73000,0,보험,nan,현금_KB_20260109_어린이보험_nan,2026-02-09 13:07:52
...,...,...,...,...,...,...,...,...,...,...,...,...
346,531,2026-03-10 00:00:00,현금,카카오뱅크,진영 학자금,single,100000,0,대출이자,nan,현금_카카오뱅크_20260310_진영 학자금_nan,2026-04-25 09:03:01
347,532,2026-03-07 00:00:00,현금,카카오뱅크,양압기,single,106800,0,기타,nan,현금_카카오뱅크_20260307_양압기_nan,2026-04-25 09:03:01
348,533,2026-03-01 00:00:00,현금,카카오뱅크,진영 용돈,single,150000,0,용돈,nan,현금_카카오뱅크_20260301_진영 용돈_nan,2026-04-25 09:03:01
349,534,2026-03-01 00:00:00,현금,카카오뱅크,진영 청약,single,100000,0,저축,nan,현금_카카오뱅크_20260301_진영 청약_nan,2026-04-25 09:03:01
